# 1. Подготовка данных

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

labels = ["positive", "negative", "neutral"]

train_pool = pd.read_csv("rusentiment_random_posts.csv")
test_pool = pd.read_csv("rusentiment_test.csv")


In [2]:
train_pool = train_pool.loc[train_pool["label"].isin(labels), ["text", "label"]].dropna().drop_duplicates("text")
test_pool = test_pool.loc[test_pool["label"].isin(labels), ["text", "label"]].dropna().drop_duplicates("text")
train_pool = train_pool.loc[~train_pool["text"].isin(test_pool["text"])]

selected, _ = train_test_split(
    train_pool, train_size=3500, stratify=train_pool["label"], random_state=42
)
train, validation = train_test_split(
    selected, train_size=3000, test_size=500, stratify=selected["label"], random_state=42
)
test, _ = train_test_split(
    test_pool, train_size=500, stratify=test_pool["label"], random_state=42
)


In [3]:
splits = {"train": train, "validation": validation, "test": test}
for name, data in splits.items():
    splits[name] = data[["text", "label"]].reset_index(drop=True)
    splits[name].to_csv(f"{name}.csv", index=False)

train, validation, test = splits["train"], splits["validation"], splits["test"]
pd.DataFrame({name: data["label"].value_counts() for name, data in splits.items()}).T


label,neutral,positive,negative
train,1634,913,453
validation,273,152,75
test,321,121,58


In [4]:
stress = pd.read_csv("stress.csv")
stress.groupby(["category", "label"]).size().unstack(fill_value=0)


label,negative,neutral,positive
category,,,
irony,10,4,6
mixed,7,6,7
negation,7,6,7
quotation,6,8,6
typos,7,6,7


# 2. Бейслайн

In [5]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(train["text"], train["label"])

baseline_test_predictions = baseline.predict(test["text"])
baseline_stress_predictions = baseline.predict(stress["text"])

baseline.classes_[baseline.class_prior_.argmax()]


'neutral'

In [6]:
from sklearn.metrics import accuracy_score

baseline_accuracy = pd.DataFrame({
    "dataset": ["test", "stress"],
    "accuracy": [
        accuracy_score(test["label"], baseline_test_predictions),
        accuracy_score(stress["label"], baseline_stress_predictions)
    ]
})
baseline_accuracy


,dataset,accuracy
0,test,0.642
1,stress,0.300


# 3. TF-IDF и LogisticRegression

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

validation_results = []
best_accuracy = -1

for min_df in [1, 2]:
    for c in [0.5, 1, 2]:
        model = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=min_df)),
            ("classifier", LogisticRegression(C=c, max_iter=1000, random_state=42))
        ])
        model.fit(train["text"], train["label"])
        score = accuracy_score(validation["label"], model.predict(validation["text"]))
        validation_results.append({"min_df": min_df, "C": c, "accuracy": score})
        if score > best_accuracy:
            best_accuracy = score
            classic_model = model

pd.DataFrame(validation_results).sort_values("accuracy", ascending=False)


,min_df,C,accuracy
4,2,1.0,0.646
5,2,2.0,0.626
1,1,1.0,0.622
2,1,2.0,0.622
3,2,0.5,0.614
0,1,0.5,0.596


In [8]:
classic_test_predictions = classic_model.predict(test["text"])
classic_stress_predictions = classic_model.predict(stress["text"])

classic_accuracy = pd.DataFrame({
    "dataset": ["test", "stress"],
    "accuracy": [
        accuracy_score(test["label"], classic_test_predictions),
        accuracy_score(stress["label"], classic_stress_predictions)
    ]
})
classic_accuracy


,dataset,accuracy
0,test,0.694
1,stress,0.380


# 4. Few-shot

In [9]:
import json

with open("few_shot_examples.json", encoding="utf-8") as file:
    few_shot_examples = json.load(file)

few_shot_prompt = "Определи тональность текста: positive, negative или neutral. Верни только JSON с единственным полем label."
for example in few_shot_examples:
    few_shot_prompt += "\nТекст: " + example["text"]
    few_shot_prompt += "\nОтвет: " + json.dumps({"label": example["label"]}, ensure_ascii=False)

print(few_shot_prompt)


Определи тональность текста: positive, negative или neutral. Верни только JSON с единственным полем label.
Текст: Еще одно успешное мероприятие в активе!
Ответ: {"label": "positive"}
Текст: А мне больше третий фильм понравился=))))))))))))))
Ответ: {"label": "positive"}
Текст: дернуть бы куда из депрессии.
Ответ: {"label": "negative"}
Текст: Без второй совы скучно(
Ответ: {"label": "negative"}
Текст: У квадратного стола отпилили один угол. Сколько теперь углов у него стало?
Ответ: {"label": "neutral"}
Текст: свалил на полигон. буду 2-го мая.
Ответ: {"label": "neutral"}


In [10]:
with open("stress_llm_few_shot_predictions.json", encoding="utf-8") as file:
    few_shot_results = json.load(file)

few_shot_stress_predictions = [result["label"] for result in few_shot_results]
pd.DataFrame(few_shot_results)


,label
0,positive
1,positive
2,positive
3,positive
4,positive
...,...
95,neutral
96,neutral
97,neutral
98,neutral


# 5. Сравнение

In [11]:
from sklearn.metrics import f1_score

with open("stress_zero_shot.json", encoding="utf-8") as file:
    zero_shot_stress_predictions = [item["label"] for item in json.load(file)]

with open("test_llm_predictions.json", encoding="utf-8") as file:
    zero_shot_test_predictions = [item["label"] for item in json.load(file)]
with open("test_llm_few_shot_predictions.json", encoding="utf-8") as file:
    few_shot_test_predictions = [item["label"] for item in json.load(file)]

predictions = {
    "Бейслайн": {"test": baseline_test_predictions, "stress": baseline_stress_predictions},
    "TF-IDF + LogisticRegression": {"test": classic_test_predictions, "stress": classic_stress_predictions},
    "LLM zero-shot": {"test": zero_shot_test_predictions, "stress": zero_shot_stress_predictions},
    "LLM few-shot": {"test": few_shot_test_predictions, "stress": few_shot_stress_predictions}
}

rows = []
for method, datasets in predictions.items():
    row = {"method": method}
    for name, data in [("test", test), ("stress", stress)]:
        predicted = datasets[name]
        row[f"{name} accuracy"] = accuracy_score(data["label"], predicted) if predicted is not None else None
        row[f"{name} macro-F1"] = f1_score(data["label"], predicted, labels=labels, average="macro", zero_division=0) if predicted is not None else None
    rows.append(row)

comparison = pd.DataFrame(rows).set_index("method")
comparison.to_csv("comparison.csv")

all_predictions = []
for name, data in [("test", test), ("stress", stress)]:
    result = data[["text", "label"]].copy()
    result.insert(0, "id", range(1, len(result) + 1))
    result.insert(0, "dataset", name)
    for column, method in [("baseline", "Бейслайн"), ("tfidf_logreg", "TF-IDF + LogisticRegression"), ("llm_zero_shot", "LLM zero-shot"), ("llm_few_shot", "LLM few-shot")]:
        result[column] = predictions[method][name]
    all_predictions.append(result)
pd.concat(all_predictions, ignore_index=True).to_csv("predictions.csv", index=False)

comparison.round(3).fillna("Нет предсказаний")


,test accuracy,test macro-F1,stress accuracy,stress macro-F1
method,,,,
Бейслайн,0.642,0.261,0.30,0.154
TF-IDF + LogisticRegression,0.694,0.501,0.38,0.313
LLM zero-shot,0.840,0.818,1.00,1.000
LLM few-shot,0.856,0.830,1.00,1.000


# 6. Время и токены

In [12]:
import time
import tiktoken

tokenizer = tiktoken.get_encoding("o200k_base")
timing_rows = []

for method, model in [("Бейслайн", baseline), ("TF-IDF + LogisticRegression", classic_model)]:
    model.predict(train["text"].iloc[:1])
    for name, data in [("test", test), ("stress", stress)]:
        start = time.perf_counter()
        model.predict(data["text"])
        elapsed = time.perf_counter() - start
        timing_rows.append({
            "method": method, "dataset": name,
            "seconds_per_text": elapsed / len(data),
            "input_tokens": None, "output_tokens": None,
        })

zero_shot_prompt = "Определи тональность текста: positive, negative или neutral. Верни только JSON с единственным полем label."
with open("chat_timing.json", encoding="utf-8") as file:
    chat_timings = json.load(file)["records"]

for record in chat_timings:
    texts = pd.read_csv(record["dataset"] + ".csv", usecols=["text"])["text"].tolist()
    prompt = few_shot_prompt if record["method"] == "LLM few-shot" else zero_shot_prompt
    visible_input = prompt + "\nТексты:\n" + json.dumps(texts, ensure_ascii=False)
    with open(record["predictions_file"], encoding="utf-8") as file:
        answers = json.load(file)
    visible_output = json.dumps(answers, ensure_ascii=False, separators=(",", ":"))
    timing_rows.append({
        "method": record["method"], "dataset": record["dataset"],
        "seconds_per_text": record["seconds"] / record["count"],
        "input_tokens": len(tokenizer.encode(visible_input)),
        "output_tokens": len(tokenizer.encode(visible_output)),
    })

timing = pd.DataFrame(timing_rows)
timing["input_tokens_per_text"] = timing.apply(
    lambda row: row["input_tokens"] / (500 if row["dataset"] == "test" else 100), axis=1
)
timing["output_tokens_per_text"] = timing.apply(
    lambda row: row["output_tokens"] / (500 if row["dataset"] == "test" else 100), axis=1
)
timing.to_csv("timing_tokens.csv", index=False)
timing.round(6)


,method,dataset,seconds_per_text,input_tokens,output_tokens,input_tokens_per_text,output_tokens_per_text
0,Бейслайн,test,0.000000,NaN,NaN,NaN,NaN
1,Бейслайн,stress,0.000000,NaN,NaN,NaN,NaN
2,TF-IDF + LogisticRegression,test,0.000007,NaN,NaN,NaN,NaN
3,TF-IDF + LogisticRegression,stress,0.000007,NaN,NaN,NaN,NaN
4,LLM zero-shot,test,0.064000,13358.0,2003.0,26.716,4.006
5,LLM zero-shot,stress,0.090000,1623.0,403.0,16.230,4.030
6,LLM few-shot,test,0.030000,13497.0,2003.0,26.994,4.006
7,LLM few-shot,stress,0.080000,1762.0,403.0,17.620,4.030
